# 02 - Test du retrieval : Precision@k, Recall@k, source hit@k, MRR

In [ ]:
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
os.environ["USE_OPENAI_EMBEDDINGS"] = "false"
os.environ["USE_LOCAL_LLM"] = "true"

In [ ]:
from src.indexing.embeddings import get_embedding_model
from src.indexing.vectorstore import load_vectorstore
from src.retrieval.retriever import create_retriever
from src.utils.config import CHROMA_PERSIST_DIR

embeddings = get_embedding_model()
vs = load_vectorstore(CHROMA_PERSIST_DIR, embeddings)
retriever = create_retriever(vs)

In [ ]:
questions = json.loads(
    Path("data/evaluation/test_questions.json").read_text(encoding="utf-8")
)["questions"]
print(f"{len(questions)} questions")

In [ ]:
import statistics


def eval_retrieval(retriever, questions, k=5):
    p1, p5, hit1, hit5, mrr = [], [], [], [], []
    for q in questions:
        docs = retriever.invoke(q["question"])
        sources = [
            d.metadata.get("document", d.metadata.get("source", "?")) for d in docs
        ]
        gold = q["expected_source"]
        hit = [s == gold for s in sources]
        p1.append(1.0 if hit[0] else 0.0)
        p5.append(sum(hit) / k)
        hit1.append(1.0 if any(hit[:1]) else 0.0)
        hit5.append(1.0 if any(hit) else 0.0)
        rr = 0.0
        for i, h in enumerate(hit):
            if h:
                rr = 1.0 / (i + 1)
                break
        mrr.append(rr)
    return {
        m: round(statistics.mean(v), 3)
        for m, v in {
            "P@1": p1,
            "P@5": p5,
            "hit@1": hit1,
            "hit@5": hit5,
            "MRR": mrr,
        }.items()
    }


eval_retrieval(retriever, questions)

In [ ]:
print(
    "Resultats identiques au fichier data/evaluation/retrieval_results.json (voir `python main.py eval-retrieval`)"
)